# Theorem 6 — posterior representation sufficiency

**Formal source:** [`../06_posterior_representation_sufficiency.md`](../06_posterior_representation_sufficiency.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
theta = np.array([-1.0, 0.0, 1.0])
posterior_a = np.array([0.5, 0.0, 0.5])
posterior_b = np.array([0.0, 1.0, 0.0])
assert theta @ posterior_a == theta @ posterior_b == 0
indicator = (abs(theta) > 0.5).astype(float)
probability_a = float(indicator @ posterior_a)
probability_b = float(indicator @ posterior_b)
assert (probability_a, probability_b) == (1.0, 0.0)
mean_only_brier = 0.25
print({"posterior_task_probabilities": [probability_a, probability_b], "posterior_brier": 0.0, "mean_only_brier": mean_only_brier})

In [ ]:
print('THEORY_DEMO_PASS::06_posterior_representation_sufficiency')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')